# Project: Build a Regression Model from Scratch

In Lesson 3 we built the full regression pipeline together — gradient descent, loss functions, feature scaling, evaluation. Now it's your turn to apply that on a fresh dataset.

The goal isn't just to get a number out of sklearn. It's to **implement gradient descent yourself**, watch it converge (or not), and understand why each step matters. The sklearn version at the end is your sanity check — if your from-scratch model gets similar results, you understood the algorithm.

Choose a dataset below, then work through the phases. Code cells are empty — that's the exercise. Use Claude Code or Codex when you get stuck.

In [ ]:
# Setup - run this first
try:
    import google.colab
    pass  # All packages pre-installed
except ImportError:
    pass

## Dataset Options

Each dataset predicts a different continuous value. Pick one that sounds interesting — you'll spend a couple of hours with it.

### 1. Auto MPG — Predict Fuel Efficiency

**What it is:** 392 cars from the late 1970s and early 1980s. Predict miles per gallon from specs like weight, horsepower, and engine displacement.

**Where:** `../../data/auto_mpg/auto-mpg.data` (included in the repo)

| Pros | Cons |
|------|------|
| Intuitive — heavier car with more horsepower = worse mileage | Small dataset (392 rows) |
| Clean relationship between features and target | A few missing values in `horsepower` (6 rows) |
| Few features — easy to reason about each one | Whitespace-separated, not CSV |
| Strong single-feature correlation (weight → mpg) | `car_name` column needs to be dropped |

**Good starting feature:** `weight` — plot it against `mpg` and you'll see a clear downward trend. Start your gradient descent here.

**Expected R² with linear regression:** ~0.70 (single feature) → ~0.82 (all features)

### 2. Wine Quality — Predict a Rating

**What it is:** ~6,500 wines rated on a scale from 3 to 9 based on chemical properties like acidity, sugar, alcohol content, and pH.

**Where:** `../../data/wine_quality/winequality-red.csv` and `winequality-white.csv` (included in the repo). Start with just red or white — don't combine them yet.

| Pros | Cons |
|------|------|
| Fun domain — wine ratings from chemistry | Target is integer (3-9), not truly continuous |
| No missing values, already numeric | No single feature dominates — correlations are weak |
| Two variants to compare (red vs white) | Most wines are rated 5 or 6 — narrow range |
| All features have clear units (g/L, pH, %) | Linear regression will plateau at modest R² |

**Good starting feature:** `alcohol` — highest correlation with quality, and it makes intuitive sense.

**Expected R² with linear regression:** ~0.20 (single feature) → ~0.35 (all features). This sounds low, but it teaches something important: some problems aren't very linear. That's a useful result, not a failure.

### 3. California Housing — Predict House Values

**What it is:** 20,640 California districts. Predict median house value from demographics like income, population, and location.

**Where:** `../../data/california_housing/housing.csv` (included in the repo)

| Pros | Cons |
|------|------|
| Large dataset — gradient descent feels real at 20k rows | You saw this dataset in L03, so it's less fresh |
| Strong income → price relationship | Has missing values (`total_bedrooms`) |
| 8 features with clear real-world meaning | Categorical column (`ocean_proximity`) needs encoding |
| Big enough to see batch vs mini-batch differences | Feature scales vary wildly (income vs population) |

**Good starting feature:** `median_income` — the single strongest predictor. You've seen this relationship in class, but now you build the model yourself.

**Expected R² with linear regression:** ~0.47 (income only) → ~0.65 (all features). The gap between single and multi-feature is large here — good motivation for adding features.

### 4. Diabetes — Predict Disease Progression

**What it is:** 442 patients. Predict a quantitative measure of disease progression one year after baseline, from 10 clinical measurements (age, sex, BMI, blood pressure, six blood serum measurements).

**Where:** Built into sklearn: `sklearn.datasets.load_diabetes()`

| Pros | Cons |
|------|------|
| No data cleaning needed — features are pre-standardized | Abstract features (s1-s6 are unnamed blood serum values) |
| Small, fast to iterate | Hard to build intuition about feature meaning |
| Classic ML benchmark dataset | Pre-standardized = you skip the scaling step |
| Good for practicing cross-validation | Small dataset can give noisy results |

**Good starting feature:** `bmi` — most intuitive and reasonably correlated with the target.

**Expected R² with linear regression:** ~0.34 (BMI only) → ~0.50 (all features)

### 5. Tips — Predict Tip Amount

**What it is:** 244 restaurant bills with the tip amount, party size, day of week, time of day, and whether anyone smoked.

**Where:** Built into seaborn: `seaborn.load_dataset('tips')`

| Pros | Cons |
|------|------|
| Tiny and fun — everyone understands tipping | Very small (244 rows) |
| Strong bill → tip relationship | Categorical features need encoding (day, time, sex, smoker) |
| Quick to iterate — results in seconds | Almost too simple for multi-feature regression |
| Great first dataset if you want a quick win | Limited feature variety |

**Good starting feature:** `total_bill` — obvious, strong linear relationship with tip.

**Expected R² with linear regression:** ~0.46 (bill only) → ~0.47 (all features). Adding features barely helps — the bill dominates everything else. That's an interesting finding on its own.

## Recommendations

**Top pick: Auto MPG.** Small, clean, intuitive features, strong linear relationships. You'll get satisfying results without fighting the data. The weight → mpg plot is one of the cleanest regression examples you'll find.

**For a quick win: Tips.** Smallest dataset, fewest features, fast iteration. Good if you want to focus purely on the gradient descent implementation without any data wrangling.

**For a real challenge: Wine Quality.** The weak correlations mean your single-feature model will be mediocre — that's the point. It forces you to think about what linear regression can and can't capture. When your R² is 0.20, is the model bad or is the problem just not very linear?

**If you want scale: California Housing.** 20k rows means gradient descent takes real iterations. You'll notice the difference between learning rates that work at 400 rows vs 20,000.

## The Phases

This project follows the same ML process from L03, but now you make the decisions:

```
1. Explore the data       →  What am I predicting? Which features matter?
2. Single-feature model   →  Pick one feature, implement gradient descent from scratch
3. Scale and improve      →  Add feature scaling, tune the learning rate
4. Multiple features      →  Extend to all features
5. Evaluate               →  How good is it? Compare to sklearn
6. Reflect                →  What worked, what didn't, what would you try next?
```

## Phase 1: Explore the Data

*Goal: Understand what you're predicting and which features might be useful.*

Before writing any model code, look at the data. This phase is quick but it shapes every decision that follows.

**Task 1.1: Load and inspect**

Load your dataset into a DataFrame. Check the shape, column names, data types, and first few rows. Answer:

- How many samples and features do you have?
- Which column is the target (what you're predicting)?
- Are there missing values? If so, how will you handle them?
- Are all features numeric, or do some need encoding?

In [ ]:
# Load data, inspect shape, head, info, missing values

**Task 1.2: Visualize relationships**

Plot each numeric feature against the target. Scatter plots are fine — you're looking for:

- Which features have a clear trend with the target? (These will make good first features)
- Are the relationships roughly linear, or curved?
- Are there outliers that might throw off your model?

Pick the feature with the strongest visible relationship. That's your starting feature for Phase 2.

In [ ]:
# Scatter plots: each feature vs target

**Task 1.3: Check correlations**

Compute the correlation between every feature and the target. This confirms (or surprises) what you saw in the scatter plots.

A correlation of 0.7+ means strong linear relationship. Under 0.3 means the feature won't help much in a linear model — though it might still help combined with others.

In [ ]:
# Correlation with target, sorted by absolute value

**Task 1.4: Train/test split**

Split your data before doing anything else. 80/20 is standard. Set a `random_state` so your results are reproducible.

From here on, **only use the training set** for building the model. The test set stays sealed until Phase 5.

In [ ]:
# Train/test split

## Phase 2: Single-Feature Gradient Descent (From Scratch)

*Goal: Implement gradient descent yourself for `y = wx + b` using one feature.*

This is the core of the project. Don't use sklearn here — write the math. You need three things:

1. A **prediction function**: `y_pred = weight * x + bias`
2. A **loss function**: MSE = mean of (prediction - actual)²
3. A **gradient computation**: how much does the loss change if we nudge weight or bias?

The gradients for MSE are:

```
grad_weight = (2/n) * sum(x * (prediction - actual))
grad_bias   = (2/n) * sum(prediction - actual)
```

If you're unsure, look back at Part 4 and Part 7 of the L03 notebook.

**Task 2.1: Write predict, loss, and gradient functions**

Three small functions. Keep them simple — no classes, no decorators, just plain Python with numpy.

In [ ]:
# def predict(x, weight, bias): ...
# def mse_loss(y_pred, y_true): ...
# def compute_gradients(x, y_true, y_pred): ...

**Task 2.2: Run gradient descent**

Initialize weight and bias to zero (or small random values). Then loop:

```
for each iteration:
    predictions = predict(x, weight, bias)
    loss = mse_loss(predictions, y_true)
    grad_w, grad_b = compute_gradients(x, y_true, predictions)
    weight = weight - learning_rate * grad_w
    bias = bias - learning_rate * grad_b
    save the loss
```

Start with `learning_rate = 0.01` and 100 iterations. Print the loss every 10 iterations to check it's decreasing.

In [ ]:
# Gradient descent loop

**Task 2.3: Plot the loss curve**

Plot loss vs iteration. You should see it drop steeply at first, then flatten. If it explodes upward, your learning rate is too high. If it barely moves, it's too low.

Also plot your learned line on the scatter plot from Phase 1. Does the fit look reasonable?

In [ ]:
# Loss curve + scatter plot with fitted line

## Phase 3: Feature Scaling

*Goal: See why scaling matters and how it changes training.*

If your feature values are large (e.g. car weight in the thousands), the gradients will be large too, and you'll need a tiny learning rate to avoid exploding. Standardization fixes this: subtract the mean, divide by the standard deviation. Every feature ends up centered around 0 with a spread of ~1.

**Important:** fit the scaler on the training set only, then apply it to both train and test. If you compute the mean/std on the full dataset, you're leaking test information into training.

**Task 3.1: Standardize your feature and re-train**

Standardize your single feature using the training set mean and std:

```python
x_mean = x_train.mean()
x_std = x_train.std()
x_train_scaled = (x_train - x_mean) / x_std
```

Re-run gradient descent on the scaled data. Try a larger learning rate (0.1 or even 0.5). Compare:

- How many iterations to converge now vs before?
- Can you use a higher learning rate without exploding?

In [ ]:
# Standardize, re-train, compare convergence

**Task 3.2: Experiment with learning rates**

Try at least 3 different learning rates on the scaled data (e.g. 0.001, 0.1, 1.0). Plot all three loss curves on the same chart.

What you should see:
- Too small: slow, barely converges in 100 iterations
- Just right: smooth, fast convergence
- Too large: loss jumps around or explodes

In [ ]:
# Compare 3+ learning rates on same plot

## Phase 4: Multiple Features

*Goal: Extend your model from one feature to all features.*

Single-feature regression is `y = w*x + b`. Multiple features is the same idea with more weights:

```
y = w₁*x₁ + w₂*x₂ + ... + wₙ*xₙ + b
```

In code, that's a dot product: `y_pred = X @ weights + bias`, where `weights` is now a vector and `X` is a matrix.

The gradient update becomes:
```
grad_weights = (2/n) * (X.T @ errors)
grad_bias    = (2/n) * errors.sum()
```

The pattern is identical — just vectors instead of scalars.

**Task 4.1: Prepare all features**

Select your numeric features. Handle missing values and encode categoricals if needed. Standardize all features (same rule: fit on train only).

In [ ]:
# Prepare and scale all features

**Task 4.2: Update your gradient descent for multiple features**

Modify your functions to work with a weight vector instead of a single weight. The predict function becomes a dot product, and the gradient is a vector.

Train on all features. Does the loss drop lower than the single-feature model? It should — more information means better predictions (usually).

In [ ]:
# Multi-feature gradient descent

**Task 4.3: Inspect the learned weights**

Plot the weight values as a bar chart with feature names on the y-axis. Which features got the largest weights (positive or negative)? Does this match your intuition from Phase 1?

Remember: since features are standardized, the weight magnitudes are directly comparable. A weight of 0.5 means "one standard deviation increase in this feature adds 0.5 to the prediction."

In [ ]:
# Feature weights bar chart

## Phase 5: Evaluate

*Goal: Measure how good your model is and compare to sklearn.*

**Task 5.1: Compute metrics on the test set**

Now unseal the test set. Scale the test features using the training mean/std (not the test set's own stats). Predict and compute:

- **RMSE** (Root Mean Squared Error) — in the same units as the target, easy to interpret
- **R²** (R-squared) — 1.0 = perfect, 0.0 = no better than predicting the mean

Compare your single-feature R² to your multi-feature R². How much did adding features help?

In [ ]:
# Evaluate on test set: RMSE, R²

**Task 5.2: Compare to sklearn**

Train `sklearn.linear_model.LinearRegression` on the same data. Compare its weights, bias, and test R² to yours.

They should be very close. If there's a gap, it likely means your gradient descent hasn't fully converged — try more iterations or a different learning rate. sklearn uses a direct mathematical solution (normal equation), not gradient descent, so it finds the exact optimum.

In [ ]:
# sklearn LinearRegression — compare weights and R²

**Task 5.3: Predictions vs actuals**

Make a scatter plot with actual values on the x-axis and predictions on the y-axis. Draw the diagonal line (perfect predictions). Points close to the line = good predictions. Points far away = errors.

Look for patterns:
- Are errors random (good) or systematic (the model is missing something)?
- Does the model struggle more at high or low values?

In [ ]:
# Scatter: actual vs predicted

## Reflection

Before closing this notebook:

1. **What happened when you didn't scale features?** How did it affect the learning rate you could use?
2. **How close did your from-scratch model get to sklearn?** If there's a gap, what might explain it?
3. **Which features mattered most?** Does the ranking match your intuition?
4. **What's the R² telling you?** If it's 0.35, that means 65% of the variance is unexplained. What information might the model be missing?
5. **What would you try next?** Polynomial features? Regularization? A non-linear model? (Spoiler: we'll get to all of these.)

The training loop you just built — predict, compute loss, compute gradients, update weights — is the same loop that trains every neural network. The model gets more complex, but the learning algorithm stays the same.

<div style="text-align: center; color: #888; font-size: 0.85em; margin-top: 40px; padding-top: 10px; border-top: 1px solid #ddd;">
© 2025 Utvecklarakademin UA Aktiebolag. All rights reserved.<br>
This material is proprietary and may not be reproduced, distributed, or shared without written permission.
</div>